# OpenMontage TTS Compare - Chatterbox vs Kokoro (Jeju / crime-ledger)
Real measurement on T4. Each model loads ONCE (singleton) then synthesizes
the same 10 narration lines. RTF = total_audio_seconds / total_generation_seconds.
Notebook is self-reporting: it always writes a manifest, capturing any
install/import failure instead of hard-crashing with no output.


In [ ]:
import os
import sys
import time
import json
import torch
from pathlib import Path

TEST_FORCE_T4 = True
GPU_BRANCH = 'unknown'
GPU_NAME = 'unknown'

if torch.cuda.is_available():
    cc = torch.cuda.get_device_properties(0).major
    GPU_NAME = torch.cuda.get_device_name(0)
    if TEST_FORCE_T4:
        if cc < 7:
            print(f'[FATAL] T4 required, got CC={cc} ({GPU_NAME}). Aborting.')
            sys.exit(1)
        GPU_BRANCH = 't4_or_better'
    elif cc >= 7:
        GPU_BRANCH = 't4_or_better'
    elif cc == 6:
        GPU_BRANCH = 'p100'
    else:
        GPU_BRANCH = 'old_gpu'
else:
    print('No GPU available.')
    sys.exit(1)

print(f'GPU: {GPU_NAME} (CC={cc}), branch={GPU_BRANCH}')


In [ ]:
import subprocess, sys

def pip_install(no_deps, *pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if no_deps:
        cmd.append('--no-deps')
    cmd.extend(pkgs)
    print(f'  $ pip install {"--no-deps " if no_deps else ""}' + ' '.join(pkgs), flush=True)
    try:
        subprocess.check_call(cmd)
        return True
    except subprocess.CalledProcessError as e:
        print(f'  PIP FAILED: {e}')
        return False

print('Installing chatterbox-tts with --no-deps (keep Kaggle numpy 1.26 / torch ABI intact)...', flush=True)
pip_install(True, 'chatterbox-tts')
print('Installing chatterbox runtime deps (--no-deps, no numpy change)...', flush=True)
pip_install(True, 'resemble-perth>=1.0.0', 'conformer==0.3.2', 'spacy-pkuseg',
              'pykakasi==2.3.0', 'pyloudnorm', 'omegaconf', 's3tokenizer',
              'librosa==0.11.0', 'gradio==6.8.0')
print('Pinning transformers/diffusers to chatterbox requirements (--no-deps)...', flush=True)
pip_install(True, 'transformers==5.2.0', 'diffusers==0.29.0')
print('Install step done.', flush=True)


In [ ]:
import torch, warnings, time, traceback
warnings.filterwarnings('ignore')
device = 'cuda'

LOAD = {}
ERRORS = []

print('Loading Chatterbox (singleton)...', flush=True)
cb = None; cb_sr = 24000
try:
    t0 = time.time()
    from chatterbox.tts import ChatterboxTTS
    cb = ChatterboxTTS.from_pretrained(device=device)
    cb_sr = int(cb.sr)
    LOAD['chatterbox'] = round(time.time() - t0, 2)
    print(f'  Chatterbox loaded in {LOAD["chatterbox"]}s, sr={cb_sr}', flush=True)
except Exception as e:
    ERRORS.append('chatterbox_load: ' + repr(e))
    print('  Chatterbox load FAILED:', repr(e), flush=True)
    traceback.print_exc()

print('Loading Kokoro (singleton)...', flush=True)
kp = None; g2p = None; kokoro_voice = 'am_puck'
try:
    t0 = time.time()
    from kokoro import KPipeline
    from misaki import en
    kp = KPipeline(lang_code='a')
    g2p = en.G2P()
    LOAD['kokoro'] = round(time.time() - t0, 2)
    print(f'  Kokoro loaded in {LOAD["kokoro"]}s', flush=True)
except Exception as e:
    ERRORS.append('kokoro_load: ' + repr(e))
    print('  Kokoro load FAILED:', repr(e), flush=True)
    traceback.print_exc()


In [ ]:
import json, os, time
from pathlib import Path
import torch

OUT = Path('/kaggle/working/outputs')
OUT.mkdir(parents=True, exist_ok=True)

SCENES = [
    ('scene_01', 'February first, 2009. Jeju Island. A childcare teacher boards a taxi at three AM.'),
    ('scene_09', 'Park drove a white NF Sonata. CCTV places a matching vehicle near the scene at the critical time.'),
    ('scene_18', 'The appellate court reviews. Same evidence. Same arguments. Same conclusion. Not guilty again.'),
    ('scene_27', 'Jeju police form a cold case team in 2016. They re-examine the body. The taxi. The timeline.'),
    ('scene_36', 'No murder weapon is found. No DNA connects Park. The fibers are not unique. The CCTV is probabilistic.'),
    ('scene_45', 'The Supreme Court explicitly noted the second taxi possibility. Reasonable doubt. The victim may have boarded a third vehicle.'),
    ('scene_54', 'Park left Jeju after the first investigation. He lived elsewhere for nine years. The case went cold.'),
    ('scene_63', "Jeju's Memories of Murder. The nickname references a Korean film about an unsolved killing. Parallels are unavoidable."),
    ('scene_72', 'The family receives no justice. No conviction. No apology. Just a Supreme Court document explaining why evidence fell short.'),
    ('scene_81', 'The court\'s reasoning is precise. Circumstantial evidence can convict. But it must exclude all reasonable alternatives.'),
]

def save_wav(path, wav, sr):
    import torchaudio as ta
    ta.save(str(path), wav.cpu(), sr)

scene_rows = []
engines = {}

if cb is not None:
    cb_aud = 0.0; cb_gen = 0.0
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    for sid, text in SCENES:
        t0 = time.time()
        try:
            wav = cb.generate(text)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            gen = time.time() - t0
            dur = int(wav.shape[-1]) / cb_sr
            p = OUT / f'{sid}_chatterbox.wav'
            save_wav(p, wav, cb_sr)
            cb_aud += dur; cb_gen += gen
            scene_rows.append({'id': sid, 'text': text, 'chatterbox': {'gen_s': round(gen,3), 'dur_s': round(dur,3), 'bytes': p.stat().st_size}, 'kokoro': None})
        except Exception as e:
            ERRORS.append(f'chatterbox_gen {sid}: ' + repr(e))
            scene_rows.append({'id': sid, 'text': text, 'chatterbox': {'error': repr(e)}, 'kokoro': None})
    engines['chatterbox'] = {'rtf': round(cb_aud / max(cb_gen,1e-6), 3), 'total_gen_s': round(cb_gen,2), 'total_audio_s': round(cb_aud,2), 'vram_peak_bytes': int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0, 'model_load_s': LOAD.get('chatterbox'), 'singleton': True, 'voice': 'default'}
else:
    for sid, text in SCENES:
        scene_rows.append({'id': sid, 'text': text, 'chatterbox': None, 'kokoro': None})

if kp is not None:
    kp_aud = 0.0; kp_gen = 0.0
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    for row in scene_rows:
        sid = row['id']; text = row['text']
        try:
            vt = kp.load_voice(kokoro_voice)
            _, tokens = g2p(text)
            t0 = time.time()
            chunks = [r.audio for r in kp.generate_from_tokens(tokens, vt)]
            if torch.cuda.is_available(): torch.cuda.synchronize()
            gen = time.time() - t0
            audio = torch.cat(chunks, dim=0)
            dur = len(audio) / 24000
            p = OUT / f'{sid}_kokoro.wav'
            save_wav(p, audio, 24000)
            kp_aud += dur; kp_gen += gen
            row['kokoro'] = {'gen_s': round(gen,3), 'dur_s': round(dur,3), 'bytes': p.stat().st_size}
        except Exception as e:
            ERRORS.append(f'kokoro_gen {sid}: ' + repr(e))
            row['kokoro'] = {'error': repr(e)}
    engines['kokoro'] = {'rtf': round(kp_aud / max(kp_gen,1e-6), 3), 'total_gen_s': round(kp_gen,2), 'total_audio_s': round(kp_aud,2), 'vram_peak_bytes': int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0, 'model_load_s': LOAD.get('kokoro'), 'singleton': True, 'voice': kokoro_voice}

manifest = {
    'project_id': 'tts-compare-jeju',
    'channel': 'crime-ledger',
    'gpu_name': GPU_NAME,
    'gpu_branch': GPU_BRANCH,
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'error': (ERRORS[0] if ERRORS else None),
    'errors': ERRORS,
    'engines': engines,
    'scenes': scene_rows,
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
